In [1]:
import numpy as np
import math
from utils import preprocess_pil, scan_raw_folders, run_preprocess

In [2]:
raw_root = "../data_raw/CUB"
out_root = "../data_ready/CUB"

classes  = np.loadtxt(f"{raw_root}/metadata/classes.txt",  dtype=str)[:, 1]
concepts = np.loadtxt(f"{raw_root}/metadata/concepts.txt", dtype=str)[:, 1]

# usecols=(0,1,2,3): [image_id, concept_id, present(0/1), certainty(1..4)]
image_concept_labels_raw = np.loadtxt(
    f"{raw_root}/metadata/image_concept.txt",
    usecols=(0, 1, 2, 3), dtype=np.int32
)
# 1-based class ids
image_class_labels = np.loadtxt(
    f"{raw_root}/metadata/image_classes.txt", dtype=np.int32
)[:, 1]

n_instances = len(image_class_labels)
n_concepts  = len(concepts)
n_classes   = len(classes)

# ---- 1) sort by (image_id, concept_id) to avoid misalignment, then reshape ----
order = np.lexsort(
    (image_concept_labels_raw[:, 1],  # concept_id
     image_concept_labels_raw[:, 0])  # image_id
)
arr = image_concept_labels_raw[order]  # shape: (N*D, 4)
present   = arr[:, 2].astype(np.int32)  # 0/1
certainty = arr[:, 3].astype(np.int32)  # 1..4

# uncertainty calibration (same as before)
map_present = np.array([0.00, 0.00, 0.50, 0.75, 1.00], dtype=np.float32)  # idx 0 unused
map_absent  = np.array([0.00, 0.00, 0.50, 0.25, 0.00], dtype=np.float32)  # idx 0 unused

calibrated_flat = np.where(
    present == 1,
    map_present[certainty],
    map_absent[certainty]
).astype(np.float32)

# reshape to (N, D)
image_concept_values = calibrated_flat.reshape(n_instances, n_concepts)

# ---- visibility-aware majority vote: ignore certainty==1 when averaging ----
certainty_mat = certainty.reshape(n_instances, n_concepts)  # (N, D)
visible_mask  = (certainty_mat != 1)                        # True means "use this entry"

class_level_concepts = np.zeros((n_classes, n_concepts), dtype=np.float32)
for c in range(n_classes):
    mask_c = (image_class_labels == (c + 1))   # 1-based labels
    if mask_c.any():
        vals_c   = image_concept_values[mask_c]         # (n_c, D)
        vis_c    = visible_mask[mask_c]                 # (n_c, D)
        # set invisible entries to NaN, then nanmean
        vals_vis = np.where(vis_c, vals_c, np.nan)
        mean_vec = np.nanmean(vals_vis, axis=0)         # (D,)
        # nanmean on all-NaN columns returns NaN → treat as 0 (not enough evidence)
        mean_vec = np.nan_to_num(mean_vec, nan=0.0)
        class_level_concepts[c] = (mean_vec >= 0.5).astype(np.float32)

# ---- keep concepts present in >= 10 classes (for CUB-200) ----
min_classes = 10 if n_classes == 200 else max(1, math.ceil(0.05 * n_classes))
occurrence_in_class = class_level_concepts.sum(axis=0)
keep_mask = (occurrence_in_class >= min_classes)

class_level_concepts_filtered = class_level_concepts[:, keep_mask]
concepts_filtered    = concepts[keep_mask]
concepts_per_image   = class_level_concepts_filtered[(image_class_labels - 1)]

print("Original concepts:", n_concepts)
print("Kept concepts:", int(keep_mask.sum()), f"(threshold: >= {min_classes} classes)")
print("class_level_concepts shape:", class_level_concepts_filtered.shape)
print("concepts_per_image shape:", concepts_per_image.shape)

# --- optional sanity checks ---
imgs = image_concept_labels_raw[:, 0]
confs = image_concept_labels_raw[:, 1]
u, cnt = np.unique(imgs, return_counts=True)
assert imgs.min() == 1 and imgs.max() == n_instances, "image_id range mismatch"
assert confs.min() == 1 and confs.max() == n_concepts, "concept_id range mismatch"
assert (cnt == n_concepts).all(), "some images do not have D concept rows"

Original concepts: 312
Kept concepts: 110 (threshold: >= 10 classes)
class_level_concepts shape: (200, 110)
concepts_per_image shape: (11788, 110)


In [3]:
run_preprocess(classes = classes, concepts = concepts_filtered, class_level_concepts = class_level_concepts_filtered, 
               raw_root = raw_root, out_root = out_root, 
               image_size = 224, shortest_side = 256, jpeg_quality = 95, save_npy = True)

Preprocessing images: 100%|███████| 11788/11788 [00:54<00:00, 218.04it/s]

[OK] Wrote 11788 images.
 - images:   ..\data_ready\CUB\images
 - metadata: ..\data_ready\CUB\metadata
 - annotations: ..\data_ready\CUB\annotations (labels_onehot.npy, concepts_per_image.npy)
(11788, 200) (11788, 110)
